# ArchiMind dataset processing (Colab driver)

Setup guide: `docs/developer/DATASET_COLAB.md`. This notebook only calls the tested
`python -m archimind.datasets` commands. Edit **only** the configuration cell.
Re-running from the top with the same `RUN_ID` resumes: finished stages are skipped.

In [ ]:
# --- Configuration (edit here only) ---
RUN_ID = "pilot-001"  # reuse the same id to resume after a disconnect
ARCHIMIND_COMMIT = "REPLACE_WITH_40_CHAR_COMMIT_SHA"  # exact commit, never a branch
REPO = "ngstephen1/ArchiMind"
WORK = "/content/work"  # fast local disk (lost when the runtime ends)
DRIVE_ROOT = "/content/drive/MyDrive/ArchiMind-data"  # persistent backup
FLOOR_GIB = 4  # stop below this much free local disk

In [ ]:
# Mount Drive and create the run layout.
import datetime
import os
import pathlib
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

RUN_DRIVE = pathlib.Path(DRIVE_ROOT) / "runs" / RUN_ID
for d in ("raw", "stages", "packs", "receipts"):
    (pathlib.Path(WORK) / d).mkdir(parents=True, exist_ok=True)
for d in ("stages", "packs", "receipts"):
    (RUN_DRIVE / d).mkdir(parents=True, exist_ok=True)
print("local work:", WORK)
print("drive backup:", RUN_DRIVE)

In [ ]:
# Load secrets into this process only. Values are never printed.
from google.colab import userdata

for name in ("GH_TOKEN", "HF_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: loaded")
    except Exception:
        print(
            f"{name}: not available (add it under the key icon and enable notebook access)"
        )

In [ ]:
# Install the pinned ArchiMind commit. Git output is hidden so the token never appears.
import re

assert re.fullmatch(r"[0-9a-f]{40}", ARCHIMIND_COMMIT), (
    "Set ARCHIMIND_COMMIT to a full commit SHA"
)
SRC = "/content/ArchiMind"
PLAIN = f"https://github.com/{REPO}.git"
AUTHED = (
    f"https://x-access-token:{os.environ.get('GH_TOKEN', '')}@github.com/{REPO}.git"
)


def _git(*args, cwd=SRC):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(
            f"git {args[0]} failed; output hidden to protect the token. Check GH_TOKEN."
        )


if not os.path.isdir(f"{SRC}/.git"):
    _git("clone", "--quiet", "--no-checkout", AUTHED, SRC, cwd="/content")
try:
    _git("remote", "set-url", "origin", AUTHED)
    _git("fetch", "--quiet", "origin", ARCHIMIND_COMMIT)
    _git("checkout", "--quiet", "--detach", ARCHIMIND_COMMIT)
finally:
    _git("remote", "set-url", "origin", PLAIN)  # never leave the token in .git/config

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", SRC],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "pydantic==2.14.1"], check=True
)
os.environ["ARCHIMIND_COMMIT"] = ARCHIMIND_COMMIT
print("installed", ARCHIMIND_COMMIT)

In [ ]:
# Helpers: run a dataset command, checkpoint to Drive, restore from Drive.
def ds(*args):
    r = subprocess.run(
        [sys.executable, "-m", "archimind.datasets", *map(str, args)],
        capture_output=True,
        text=True,
    )
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-4000:])
        raise RuntimeError(f"archimind.datasets {args[0]} exited {r.returncode}")


def checkpoint():
    for d in ("stages", "packs", "receipts"):
        ds("sync", f"{WORK}/{d}", RUN_DRIVE / d)


def restore():
    for d in ("stages", "packs", "receipts"):
        ds("sync", RUN_DRIVE / d, f"{WORK}/{d}")

In [ ]:
# Restore earlier progress for this RUN_ID (no-op on a fresh run).
restore()

In [ ]:
# Environment receipt and free-space check (fails below FLOOR_GIB).
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ds(
    "env",
    "--data-root",
    WORK,
    "--floor-gib",
    FLOOR_GIB,
    "--out",
    f"{WORK}/receipts/env-{stamp}.json",
)
checkpoint()

In [ ]:
# Self-check: build and verify a synthetic pack end to end, then checkpoint it.
ds("fixture-pack", f"{WORK}/packs/synthetic-selfcheck")
ds("verify-pack", f"{WORK}/packs/synthetic-selfcheck")
checkpoint()

## Stages

P1 and later add cells here (`probe`, then SSR, FUTURE and MIDI processing, then `pack`).
Each stage cell ends with `checkpoint()`.

In [ ]:
# Finalize before closing the tab: checkpoint, then make Drive finish writing.
checkpoint()
drive.flush_and_unmount()
print("Drive flushed. Safe to close.")